# Genetic Algorithm for Route Optimization

## ✨ Features:
- 🧬 Genetic Algorithm for solving TSP (Traveling Salesman Problem)
- 🗺️ Uses OSRM for real driving distances
- 📊 Minimizes total travel distance
- 🎯 Integrates with K-means cluster output
- 🎨 Interactive route visualization with Folium
- 💾 Save/load optimized routes

## 🚀 Quick Start:
```python
# Load cluster from K-means
day_data = load_cluster_for_day('clusters_output.json', day_number=1)

# Optimize route
result = optimize_route_ga(day_data['coordinates'], day_data['names'])

# Visualize
result['map']  # Display interactive map
```

In [1]:
# Import required libraries
import numpy as np
import random
import json
import requests
from math import radians, cos, sin, asin, sqrt
import matplotlib.pyplot as plt
import folium
from datetime import datetime
import copy

# Set random seed for reproducibility
random.seed(42)
np.random.seed(42)

print('✅ Libraries loaded successfully!')

✅ Libraries loaded successfully!


## 📏 Distance Calculation Functions

Multiple methods with automatic fallback:
1. **OSRM** - Real driving distances (primary)
2. **Haversine** - Geographic distance (fallback)

In [ ]:
def haversine_distance(coord1, coord2):
    """
    Calculate the great-circle distance between two points on Earth.
    
    Args:
        coord1: [latitude, longitude]
        coord2: [latitude, longitude]
    
    Returns:
        Distance in kilometers
    """
    lat1, lon1 = coord1
    lat2, lon2 = coord2
    
    # Convert to radians
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    
    # Haversine formula
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    
    # Radius of Earth in kilometers
    R = 6371.0
    
    return R * c


def get_osrm_distance_matrix(coordinates, use_osrm=True, osrm_url="http://localhost:5000"):
    """
    Get distance matrix using OSRM Table API or Haversine fallback.
    
    Args:
        coordinates: List of [lat, lon] pairs
        use_osrm: If True, try OSRM first
        osrm_url: OSRM server URL
    
    Returns:
        2D numpy array of distances in kilometers
    """
    n = len(coordinates)
    
    if use_osrm:
        try:
            # Format coordinates for OSRM (lon,lat format)
            coords_str = ';'.join([f"{lon},{lat}" for lat, lon in coordinates])
            url = f"{osrm_url}/table/v1/driving/{coords_str}?annotations=distance"
            
            response = requests.get(url, timeout=10)
            
            if response.status_code == 200:
                data = response.json()
                if data['code'] == 'Ok':
                    # Convert meters to kilometers
                    distances = np.array(data['distances']) / 1000.0
                    print(f"✅ OSRM distance matrix retrieved ({n}x{n})")
                    return distances
            
            print(f"⚠️ OSRM failed (status {response.status_code}), falling back to Haversine")
        
        except Exception as e:
            print(f"⚠️ OSRM error: {e}, falling back to Haversine")
    
    # Fallback to Haversine
    print(f"📍 Using Haversine distance calculation")
    distances = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if i != j:
                distances[i][j] = haversine_distance(coordinates[i], coordinates[j])
    
    return distances


def calculate_route_distance(route, distance_matrix):
    """
    Calculate total distance for a route.
    
    Args:
        route: List of location indices [2, 5, 1, 4, 3, 0]
        distance_matrix: 2D array of distances
    
    Returns:
        Total distance in kilometers
    """
    total_distance = 0.0
    
    for i in range(len(route) - 1):
        from_idx = route[i]
        to_idx = route[i + 1]
        total_distance += distance_matrix[from_idx][to_idx]
    
    return total_distance


print('✅ Distance calculation functions loaded')

## 🧬 Genetic Algorithm Core Components

In [ ]:
# GA Parameters
DEFAULT_GA_PARAMS = {
    'population_size': 100,
    'max_generations': 50,
    'crossover_rate': 0.8,
    'mutation_rate': 0.2,
    'tournament_size': 5,
    'elite_count': 2,
}

print('✅ GA parameters configured')
for key, value in DEFAULT_GA_PARAMS.items():
    print(f'   {key}: {value}')

In [ ]:
def create_random_route(n_locations):
    """
    Create a random route (permutation of location indices).
    
    Args:
        n_locations: Number of locations to visit
    
    Returns:
        Random permutation [0, 1, 2, ...] shuffled
    """
    route = list(range(n_locations))
    random.shuffle(route)
    return route


def initialize_population(n_locations, population_size):
    """
    Create initial population of random routes.
    
    Args:
        n_locations: Number of locations
        population_size: Number of chromosomes in population
    
    Returns:
        List of random routes
    """
    return [create_random_route(n_locations) for _ in range(population_size)]


def fitness(route, distance_matrix):
    """
    Calculate fitness of a route (lower distance = better fitness).
    
    Args:
        route: List of location indices
        distance_matrix: 2D array of distances
    
    Returns:
        Fitness score (negative distance for maximization)
    """
    distance = calculate_route_distance(route, distance_matrix)
    # Return negative distance so we can maximize
    return -distance


def evaluate_population(population, distance_matrix):
    """
    Evaluate fitness for entire population.
    
    Returns:
        List of fitness scores
    """
    return [fitness(route, distance_matrix) for route in population]


print('✅ Initialization and fitness functions loaded')

In [ ]:
def tournament_selection(population, fitnesses, tournament_size=5):
    """
    Select a parent using tournament selection.
    
    Args:
        population: List of routes
        fitnesses: List of fitness scores
        tournament_size: Number of individuals in tournament
    
    Returns:
        Selected route (parent)
    """
    # Random indices for tournament
    tournament_indices = random.sample(range(len(population)), tournament_size)
    
    # Find best in tournament
    best_idx = max(tournament_indices, key=lambda i: fitnesses[i])
    
    return copy.deepcopy(population[best_idx])


print('✅ Selection function loaded')

In [ ]:
def pmx_crossover(parent1, parent2):
    """
    Partially Mapped Crossover (PMX) - preserves permutation validity.
    
    Args:
        parent1: First parent route
        parent2: Second parent route
    
    Returns:
        Two child routes
    """
    size = len(parent1)
    
    # Choose two random crossover points
    cx_point1 = random.randint(0, size - 2)
    cx_point2 = random.randint(cx_point1 + 1, size - 1)
    
    # Initialize children as copies of parents
    child1 = [-1] * size
    child2 = [-1] * size
    
    # Copy the segment between crossover points
    child1[cx_point1:cx_point2] = parent1[cx_point1:cx_point2]
    child2[cx_point1:cx_point2] = parent2[cx_point1:cx_point2]
    
    # Fill remaining positions using PMX logic
    def fill_child(child, parent_source, parent_dest, cx1, cx2):
        for i in range(size):
            if i < cx1 or i >= cx2:
                # Find value from parent_source
                value = parent_source[i]
                
                # Check if value already in segment
                while value in child[cx1:cx2]:
                    # Find position of value in parent_source
                    idx = parent_source.index(value)
                    # Get corresponding value from parent_dest
                    value = parent_dest[idx]
                
                child[i] = value
    
    fill_child(child1, parent2, parent1, cx_point1, cx_point2)
    fill_child(child2, parent1, parent2, cx_point1, cx_point2)
    
    return child1, child2


print('✅ Crossover function (PMX) loaded')

In [ ]:
def swap_mutation(route, mutation_rate=0.2):
    """
    Swap mutation - randomly swap two positions in the route.
    
    Args:
        route: Route to mutate
        mutation_rate: Probability of mutation
    
    Returns:
        Mutated route
    """
    route = copy.deepcopy(route)
    
    if random.random() < mutation_rate:
        # Choose two random positions
        idx1, idx2 = random.sample(range(len(route)), 2)
        # Swap
        route[idx1], route[idx2] = route[idx2], route[idx1]
    
    return route


print('✅ Mutation function loaded')

## 🔄 Main GA Loop

In [ ]:
def genetic_algorithm(distance_matrix, params=None):
    """
    Main Genetic Algorithm loop.
    
    Args:
        distance_matrix: 2D numpy array of distances
        params: Dict of GA parameters (optional)
    
    Returns:
        dict: {
            'best_route': [2, 5, 1, 4, 3, 0],
            'best_distance': 45.3,
            'fitness_history': [...],
            'generations': 50
        }
    """
    # Use default params if not provided
    if params is None:
        params = DEFAULT_GA_PARAMS.copy()
    
    n_locations = len(distance_matrix)
    
    # Initialize population
    population = initialize_population(n_locations, params['population_size'])
    
    # Track best solution
    best_route = None
    best_distance = float('inf')
    fitness_history = []
    
    print(f"🧬 Starting GA evolution...")
    print(f"   Population: {params['population_size']}")
    print(f"   Generations: {params['max_generations']}")
    print(f"   Locations: {n_locations}\n")
    
    # Evolution loop
    for generation in range(params['max_generations']):
        # Evaluate fitness
        fitnesses = evaluate_population(population, distance_matrix)
        
        # Track best
        best_idx = np.argmax(fitnesses)
        gen_best_distance = -fitnesses[best_idx]  # Convert back to positive
        
        if gen_best_distance < best_distance:
            best_distance = gen_best_distance
            best_route = copy.deepcopy(population[best_idx])
        
        fitness_history.append(best_distance)
        
        # Print progress every 10 generations
        if generation % 10 == 0 or generation == params['max_generations'] - 1:
            print(f"   Gen {generation:3d}: Best distance = {best_distance:.2f} km")
        
        # Create next generation
        new_population = []
        
        # Elitism: keep best individuals
        elite_indices = np.argsort(fitnesses)[-params['elite_count']:]
        for idx in elite_indices:
            new_population.append(copy.deepcopy(population[idx]))
        
        # Generate rest of population
        while len(new_population) < params['population_size']:
            # Selection
            parent1 = tournament_selection(population, fitnesses, params['tournament_size'])
            parent2 = tournament_selection(population, fitnesses, params['tournament_size'])
            
            # Crossover
            if random.random() < params['crossover_rate']:
                child1, child2 = pmx_crossover(parent1, parent2)
            else:
                child1, child2 = copy.deepcopy(parent1), copy.deepcopy(parent2)
            
            # Mutation
            child1 = swap_mutation(child1, params['mutation_rate'])
            child2 = swap_mutation(child2, params['mutation_rate'])
            
            # Add to new population
            new_population.append(child1)
            if len(new_population) < params['population_size']:
                new_population.append(child2)
        
        population = new_population
    
    print(f"\n✅ Evolution complete!")
    print(f"   Best distance: {best_distance:.2f} km")
    
    return {
        'best_route': best_route,
        'best_distance': best_distance,
        'fitness_history': fitness_history,
        'generations': params['max_generations']
    }


print('✅ Main GA loop loaded')

## 🎯 High-Level Optimization Function

In [ ]:
def optimize_route_ga(coordinates, location_names=None, use_osrm=True, params=None, verbose=True):
    """
    High-level function to optimize a route using GA.
    
    Args:
        coordinates: List of [lat, lon] pairs
        location_names: Optional list of location names
        use_osrm: If True, use OSRM for distances
        params: GA parameters (optional)
        verbose: Print progress
    
    Returns:
        dict with optimized route and metadata
    """
    n = len(coordinates)
    
    # Edge cases
    if n == 0:
        return {
            'status': 'empty',
            'message': 'No locations provided',
            'optimized_order': [],
            'total_distance_km': 0.0
        }
    
    if n == 1:
        return {
            'status': 'single',
            'message': 'Only one location, no optimization needed',
            'optimized_order': [0],
            'coordinates': coordinates,
            'location_names': location_names,
            'total_distance_km': 0.0
        }
    
    if n == 2:
        dist = haversine_distance(coordinates[0], coordinates[1])
        return {
            'status': 'two_locations',
            'message': 'Only two locations, order is trivial',
            'optimized_order': [0, 1],
            'coordinates': coordinates,
            'location_names': location_names,
            'total_distance_km': dist
        }
    
    if verbose:
        print(f"\n{'='*70}")
        print(f"🗺️  ROUTE OPTIMIZATION WITH GENETIC ALGORITHM")
        print(f"{'='*70}")
        print(f"📍 Locations: {n}")
    
    # Get distance matrix
    distance_matrix = get_osrm_distance_matrix(coordinates, use_osrm=use_osrm)
    
    # Calculate initial (unoptimized) distance
    original_order = list(range(n))
    original_distance = calculate_route_distance(original_order, distance_matrix)
    
    if verbose:
        print(f"\n📏 Original distance: {original_distance:.2f} km\n")
    
    # Run GA
    ga_result = genetic_algorithm(distance_matrix, params)
    
    # Calculate improvement
    improvement_pct = ((original_distance - ga_result['best_distance']) / original_distance) * 100
    
    if verbose:
        print(f"\n📊 Results:")
        print(f"   Original order: {original_order}")
        print(f"   Optimized order: {ga_result['best_route']}")
        print(f"   Distance reduction: {improvement_pct:.1f}%")
        print(f"{'='*70}\n")
    
    # Build route sequence
    route_sequence = []
    for idx in ga_result['best_route']:
        route_sequence.append({
            'index': idx,
            'name': location_names[idx] if location_names and idx < len(location_names) else f"Location {idx}",
            'coordinates': coordinates[idx]
        })
    
    return {
        'status': 'optimized',
        'coordinates': coordinates,
        'location_names': location_names,
        'original_order': original_order,
        'optimized_order': ga_result['best_route'],
        'original_distance_km': original_distance,
        'optimized_distance_km': ga_result['best_distance'],
        'improvement_percent': improvement_pct,
        'route_sequence': route_sequence,
        'fitness_history': ga_result['fitness_history'],
        'distance_matrix': distance_matrix
    }


print('✅ High-level optimization function loaded')

## �� Integration with K-Means Clustering

In [ ]:
def load_cluster_for_day(cluster_filename, day_number):
    """
    Load cluster data for a specific day from K-means output.
    
    Args:
        cluster_filename: JSON file from K-means clustering
        day_number: Day number (1, 2, 3, ...)
    
    Returns:
        dict with coordinates and names for that day
    """
    import os
    
    if not os.path.exists(cluster_filename):
        raise FileNotFoundError(f"Cluster file not found: {cluster_filename}")
    
    with open(cluster_filename, 'r') as f:
        data = json.load(f)
    
    # Convert cluster keys to integers
    clusters = {int(k): v for k, v in data['clusters'].items()}
    
    if day_number not in clusters:
        raise ValueError(f"Day {day_number} not found. Available: {list(clusters.keys())}")
    
    # Get location indices for this day
    location_indices = clusters[day_number]
    all_coords = data['coordinates']
    
    # Extract coordinates for this day
    day_coords = [all_coords[i] for i in location_indices]
    
    # Get names if available
    names = None
    if 'location_names' in data:
        names = [data['location_names'][i] for i in location_indices]
    
    print(f"✅ Loaded Day {day_number} from {cluster_filename}")
    print(f"   Locations: {len(day_coords)}")
    
    return {
        'day_number': day_number,
        'coordinates': day_coords,
        'location_names': names,
        'original_indices': location_indices,
        'num_locations': len(day_coords)
    }


def optimize_all_days(cluster_filename, use_osrm=True, params=None):
    """
    Optimize routes for all days in a cluster file.
    
    Args:
        cluster_filename: JSON file from K-means
        use_osrm: Use OSRM for distances
        params: GA parameters
    
    Returns:
        dict: {day_number: optimized_route_result}
    """
    with open(cluster_filename, 'r') as f:
        data = json.load(f)
    
    clusters = {int(k): v for k, v in data['clusters'].items()}
    num_days = len(clusters)
    
    print(f"\n{'='*70}")
    print(f"🗓️  OPTIMIZING MULTI-DAY TRIP")
    print(f"{'='*70}")
    print(f"Total days: {num_days}\n")
    
    all_results = {}
    
    for day in sorted(clusters.keys()):
        print(f"\n--- Optimizing Day {day} ---\n")
        
        day_data = load_cluster_for_day(cluster_filename, day)
        
        if day_data['num_locations'] <= 2:
            print(f"⏭️  Skipping optimization (only {day_data['num_locations']} locations)\n")
            all_results[day] = {
                'status': 'skipped',
                'reason': 'too few locations',
                'num_locations': day_data['num_locations']
            }
            continue
        
        result = optimize_route_ga(
            day_data['coordinates'],
            day_data['location_names'],
            use_osrm=use_osrm,
            params=params
        )
        
        result['day_number'] = day
        all_results[day] = result
    
    print(f"\n{'='*70}")
    print(f"✅ ALL DAYS OPTIMIZED")
    print(f"{'='*70}\n")
    
    return all_results


def save_optimized_routes(results, filename="optimized_routes.json"):
    """
    Save optimized routes to JSON file.
    
    Args:
        results: Results from optimize_all_days() or single day
        filename: Output filename
    """
    # Convert numpy arrays to lists for JSON serialization
    def convert_to_serializable(obj):
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        elif isinstance(obj, dict):
            return {k: convert_to_serializable(v) for k, v in obj.items()}
        elif isinstance(obj, list):
            return [convert_to_serializable(item) for item in obj]
        else:
            return obj
    
    data = {
        'timestamp': datetime.now().isoformat(),
        'results': convert_to_serializable(results)
    }
    
    with open(filename, 'w') as f:
        json.dump(data, f, indent=2)
    
    print(f"💾 Saved optimized routes to: {filename}")


print('✅ Integration functions loaded')

## 🎨 Visualization Functions

In [ ]:
def visualize_route_on_map(result, show_original=False):
    """
    Visualize optimized route on an interactive map.
    
    Args:
        result: Result dict from optimize_route_ga()
        show_original: If True, also show original route
    
    Returns:
        folium.Map object
    """
    coordinates = result['coordinates']
    optimized_order = result['optimized_order']
    names = result.get('location_names')
    
    # Calculate map center
    center_lat = np.mean([coord[0] for coord in coordinates])
    center_lon = np.mean([coord[1] for coord in coordinates])
    
    # Create map
    m = folium.Map(location=[center_lat, center_lon], zoom_start=11, tiles='OpenStreetMap')
    
    # Add optimized route
    optimized_coords = [coordinates[i] for i in optimized_order]
    
    # Draw route line
    folium.PolyLine(
        optimized_coords,
        color='blue',
        weight=3,
        opacity=0.8,
        popup='Optimized Route'
    ).add_to(m)
    
    # Add markers
    for i, idx in enumerate(optimized_order):
        coord = coordinates[idx]
        name = names[idx] if names and idx < len(names) else f"Location {idx}"
        
        # Different icon for start
        if i == 0:
            icon = folium.Icon(color='green', icon='play', prefix='fa')
            label = 'START'
        elif i == len(optimized_order) - 1:
            icon = folium.Icon(color='red', icon='stop', prefix='fa')
            label = 'END'
        else:
            icon = folium.Icon(color='blue', icon='circle', prefix='fa')
            label = f'Stop {i}'
        
        popup_html = f"""
        <div style="font-family: Arial; width: 200px;">
            <h4 style="margin: 0;">{label}</h4>
            <p style="margin: 5px 0;"><b>{name}</b></p>
            <p style="margin: 5px 0; font-size: 11px;">
                Position in route: {i + 1}/{len(optimized_order)}<br>
                Original index: {idx}<br>
                📍 {coord[0]:.6f}, {coord[1]:.6f}
            </p>
        </div>
        """
        
        folium.Marker(
            location=coord,
            popup=folium.Popup(popup_html, max_width=250),
            icon=icon,
            tooltip=f"{i+1}. {name}"
        ).add_to(m)
    
    # Add legend
    legend_html = f'''
    <div style="position: fixed; 
                top: 10px; left: 50px; width: 280px;
                background-color: white; z-index: 1000;
                border: 2px solid grey; border-radius: 5px;
                padding: 10px; font-family: Arial;">
        <h4 style="margin: 0 0 10px 0;">🧬 Optimized Route</h4>
        <p style="margin: 5px 0; font-size: 12px;">
            📍 Locations: {len(coordinates)}<br>
            📏 Distance: {result.get('optimized_distance_km', 0):.2f} km<br>
            📊 Improvement: {result.get('improvement_percent', 0):.1f}%
        </p>
    </div>
    '''
    m.get_root().html.add_child(folium.Element(legend_html))
    
    return m


def plot_convergence(result):
    """
    Plot GA convergence (fitness over generations).
    
    Args:
        result: Result dict from optimize_route_ga()
    """
    fitness_history = result.get('fitness_history', [])
    
    if not fitness_history:
        print("No fitness history available")
        return
    
    plt.figure(figsize=(10, 5))
    plt.plot(fitness_history, 'b-', linewidth=2)
    plt.xlabel('Generation', fontsize=12)
    plt.ylabel('Best Distance (km)', fontsize=12)
    plt.title('GA Convergence - Distance Minimization', fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print(f"\n📈 Convergence Analysis:")
    print(f"   Starting distance: {fitness_history[0]:.2f} km")
    print(f"   Final distance: {fitness_history[-1]:.2f} km")
    print(f"   Improvement: {((fitness_history[0] - fitness_history[-1])/fitness_history[0]*100):.1f}%")


print('✅ Visualization functions loaded')

## 📚 Usage Examples

### Example 1: Basic Route Optimization with Test Data

In [ ]:
# Sample Goa coordinates (real tourist attractions)
test_coordinates = [
    [15.4909, 73.8278],  # Fort Aguada
    [15.2993, 74.1240],  # Dudhsagar Falls
    [15.5516, 73.7555],  # Chapora Fort
    [15.4780, 73.8286],  # Calangute Beach
    [15.6333, 73.7361],  # Arambol Beach
    [15.5514, 73.7594],  # Vagator Beach
    [15.6000, 73.7619],  # Morjim Beach
]

test_names = [
    "Fort Aguada",
    "Dudhsagar Falls",
    "Chapora Fort",
    "Calangute Beach",
    "Arambol Beach",
    "Vagator Beach",
    "Morjim Beach"
]

print("🧪 Testing with 7 Goa tourist attractions...\n")

In [ ]:
# Optimize the route
result = optimize_route_ga(
    coordinates=test_coordinates,
    location_names=test_names,
    use_osrm=True,  # Use OSRM for real driving distances
    params={
        'population_size': 100,
        'max_generations': 50,
        'crossover_rate': 0.8,
        'mutation_rate': 0.2,
        'tournament_size': 5,
        'elite_count': 2
    }
)

In [ ]:
# Visualize the optimized route
route_map = visualize_route_on_map(result)
route_map

In [ ]:
# Plot convergence
plot_convergence(result)

### Example 2: Integration with K-Means Clustering

Load cluster data from K-means and optimize each day's route.

In [ ]:
# First, check if we have cluster data
import os

cluster_file = "tourist_clusters.json"

if os.path.exists(cluster_file):
    print(f"✅ Found cluster file: {cluster_file}")
    
    # Load metadata
    with open(cluster_file, 'r') as f:
        cluster_data = json.load(f)
    
    print(f"\nCluster Metadata:")
    print(f"  Trip ID: {cluster_data.get('trip_id')}")
    print(f"  Number of days: {cluster_data.get('num_clusters')}")
    print(f"  Total locations: {len(cluster_data.get('coordinates', []))}")
    print(f"  Created: {cluster_data.get('timestamp')}")
else:
    print(f"⚠️  Cluster file not found: {cluster_file}")
    print(f"   Run kmeans_clustering.ipynb first to generate cluster data")

In [ ]:
# Optimize a single day
if os.path.exists(cluster_file):
    day_data = load_cluster_for_day(cluster_file, day_number=1)
    
    day1_result = optimize_route_ga(
        coordinates=day_data['coordinates'],
        location_names=day_data['location_names'],
        use_osrm=True
    )
    
    # Visualize Day 1 route
    day1_map = visualize_route_on_map(day1_result)
    display(day1_map)
else:
    print("Skipping - no cluster file")

### Example 3: Optimize All Days at Once

In [ ]:
# Optimize all days in the trip
if os.path.exists(cluster_file):
    all_results = optimize_all_days(cluster_file, use_osrm=True)
    
    # Save results
    save_optimized_routes(all_results, "optimized_routes.json")
    
    # Summary statistics
    print("\n📊 SUMMARY:")
    for day, result in sorted(all_results.items()):
        if result.get('status') == 'skipped':
            print(f"   Day {day}: Skipped ({result.get('reason')})")
        else:
            dist = result.get('optimized_distance_km', 0)
            imp = result.get('improvement_percent', 0)
            print(f"   Day {day}: {dist:.1f} km (improved {imp:.1f}%)")
else:
    print("Skipping - no cluster file")

### Example 4: Compare with OSRM Built-in TSP Solver

OSRM has a `/trip` endpoint with built-in TSP solver. Let's compare!

In [ ]:
def osrm_trip_solver(coordinates):
    """
    Use OSRM's built-in TSP solver for comparison.
    
    Args:
        coordinates: List of [lat, lon]
    
    Returns:
        dict with route and distance
    """
    # OSRM uses lon,lat format
    coords_str = ';'.join([f"{lon},{lat}" for lat, lon in coordinates])
    url = f"{OSRM_BASE_URL}/trip/v1/driving/{coords_str}?source=first&roundtrip=false"
    
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        data = response.json()
        
        if data['code'] == 'Ok':
            # Extract waypoint order
            waypoints = data['waypoints']
            order = [wp['waypoint_index'] for wp in waypoints]
            
            # Extract distance (meters -> km)
            distance_km = data['trips'][0]['distance'] / 1000
            
            return {
                'order': order,
                'distance_km': distance_km,
                'success': True
            }
        else:
            return {'success': False, 'error': data.get('message', 'Unknown error')}
    
    except Exception as e:
        return {'success': False, 'error': str(e)}


# Compare GA vs OSRM
print("�� COMPARISON: Genetic Algorithm vs OSRM Built-in TSP\n")
print("="*60)

# Our GA solution
ga_result = optimize_route_ga(test_coordinates, test_names, use_osrm=True)

# OSRM's solution
osrm_result = osrm_trip_solver(test_coordinates)

if osrm_result['success']:
    print(f"\n📊 Results:")
    print(f"   GA Distance:   {ga_result['optimized_distance_km']:.2f} km")
    print(f"   OSRM Distance: {osrm_result['distance_km']:.2f} km")
    
    diff = abs(ga_result['optimized_distance_km'] - osrm_result['distance_km'])
    diff_pct = (diff / osrm_result['distance_km']) * 100
    
    print(f"\n   Difference: {diff:.2f} km ({diff_pct:.1f}%)")
    
    if ga_result['optimized_distance_km'] < osrm_result['distance_km']:
        print(f"   🏆 GA wins!")
    elif ga_result['optimized_distance_km'] > osrm_result['distance_km']:
        print(f"   🏆 OSRM wins!")
    else:
        print(f"   🤝 Tie!")
else:
    print(f"⚠️  OSRM comparison failed: {osrm_result.get('error')}")

print("\n" + "="*60)

### Example 5: Edge Case Testing

In [ ]:
# Test edge cases
print("🧪 Testing Edge Cases\n")
print("="*60)

# Empty route
print("\n1. Empty route (0 locations):")
try:
    result = optimize_route_ga([], [])
    print(f"   Result: {result}")
except Exception as e:
    print(f"   ✅ Handled: {e}")

# Single location
print("\n2. Single location (1 POI):")
result = optimize_route_ga([test_coordinates[0]], [test_names[0]])
print(f"   Order: {result['optimized_order']}")
print(f"   Distance: {result['optimized_distance_km']} km")

# Two locations
print("\n3. Two locations (2 POIs):")
result = optimize_route_ga(test_coordinates[:2], test_names[:2])
print(f"   Order: {result['optimized_order']}")
print(f"   Distance: {result['optimized_distance_km']:.2f} km")

# Fallback to Haversine (OSRM disabled)
print("\n4. Haversine fallback (OSRM disabled):")
result = optimize_route_ga(test_coordinates[:4], test_names[:4], use_osrm=False)
print(f"   Order: {result['optimized_order']}")
print(f"   Distance: {result['optimized_distance_km']:.2f} km")
print(f"   Distance source: Haversine")

print("\n" + "="*60)
print("✅ All edge cases handled correctly")

## 🎯 Complete Workflow Example

This shows the full pipeline:
1. Load K-means clusters
2. Optimize each day
3. Save results
4. Visualize all days

In [ ]:
if os.path.exists(cluster_file):
    print("🚀 COMPLETE WORKFLOW\n")
    print("="*70)
    
    # Step 1: Optimize all days
    print("\nStep 1: Optimizing all days...")
    all_results = optimize_all_days(cluster_file, use_osrm=True)
    
    # Step 2: Save results
    print("\nStep 2: Saving optimized routes...")
    save_optimized_routes(all_results, "optimized_routes.json")
    
    # Step 3: Visualize each day
    print("\nStep 3: Generating visualizations...\n")
    for day, result in sorted(all_results.items()):
        if result.get('status') != 'skipped':
            print(f"📍 Day {day} Map:")
            day_map = visualize_route_on_map(result)
            display(day_map)
            
            # Save map to HTML
            map_file = f"day_{day}_route.html"
            day_map.save(map_file)
            print(f"   Saved to: {map_file}\n")
    
    # Step 4: Overall statistics
    print("\n" + "="*70)
    print("📊 TRIP SUMMARY\n")
    total_distance = sum(
        r.get('optimized_distance_km', 0) 
        for r in all_results.values() 
        if r.get('status') != 'skipped'
    )
    total_improvement = np.mean([
        r.get('improvement_percent', 0) 
        for r in all_results.values() 
        if r.get('status') != 'skipped'
    ])
    
    print(f"Total optimized distance: {total_distance:.1f} km")
    print(f"Average improvement per day: {total_improvement:.1f}%")
    print(f"\n✅ Workflow complete!")
    print("="*70)
else:
    print("⚠️  Run kmeans_clustering.ipynb first to generate cluster data")

## 🎓 Next Steps

### What you can do:

1. **Tune GA Parameters**
   - Increase population size for better solutions
   - Increase generations for more convergence
   - Adjust mutation rate (higher = more exploration)

2. **Add Constraints** (Future)
   - Opening/closing times
   - Visit duration estimates
   - Lunch breaks
   - Start/end at hotel

3. **Database Integration**
   - Use pre-cached distance matrix from PostgreSQL
   - Store optimized routes in database
   - Link with user preferences

4. **API Integration**
   - Create FastAPI endpoint `/optimize-route`
   - Accept cluster data via API
   - Return optimized routes as JSON

5. **Advanced Features**
   - Multi-objective optimization (distance + time + cost)
   - User preference weighting
   - Real-time traffic data
   - Alternative route suggestions

### Files Created:
- `optimized_routes.json` - Saved optimized routes
- `day_N_route.html` - Interactive maps for each day

### Related Notebooks:
- `kmeans_clustering.ipynb` - Create multi-day clusters first
- `module1_executed.ipynb` - Data exploration and analysis